# Affine Flow (e.g. an analytic flow)

Goals:
1. Understand the 2D affine transform used by the flow.
2. Derive the Jacobian and its determinant.
3. Connect the change-of-variables formula to the negative log-likelihood used in training.


In [ ]:
import math
import numpy as np
import torch
import torch.nn as nn
import matplotlib.pyplot as plt
from matplotlib.animation import FuncAnimation, PillowWriter


# ------------------------------------------------------------
# 1. Synthetic 2D Gaussian dataset
# ------------------------------------------------------------
def make_gaussian_data(n=2000, mu=None, cov=None, device="cpu"):
    if mu is None:
        mu = torch.tensor([2.0, -1.0], dtype=torch.float32, device=device)
    if cov is None:
        cov = torch.tensor([[3.0, 1.2],
                            [1.2, 1.0]], dtype=torch.float32, device=device)

    dist = torch.distributions.MultivariateNormal(mu, covariance_matrix=cov)
    x = dist.sample((n,))
    return x, mu, cov


## Exercise 1: Write The Transform Explicitly

The flow maps a latent variable $z \in \mathbb{R}^2$ to a data point $x \in \mathbb{R}^2$ using

$$x = Az + \mu,$$

with

$$A = \begin{bmatrix} e^{s_1} & 0 \\ l_{21} & e^{s_2} \end{bmatrix}, \qquad \mu = \begin{bmatrix} \mu_1 \\ \mu_2 \end{bmatrix}. $$

Written componentwise, the transform is

$$x_1 = e^{s_1} z_1 + \mu_1,$$
$$x_2 = l_{21} z_1 + e^{s_2} z_2 + \mu_2.$$


In [ ]:
class AffineFlow2D(nn.Module):
    """
    x = A z + mu
    z = A^{-1}(x - mu)

    A is parameterized as lower triangular with positive diagonal:
        A = [[exp(s1), 0],
             [l21,     exp(s2)]]
    """
    def __init__(self):
        super().__init__()

        # Shift parameter
        self.mu = nn.Parameter(torch.zeros(2))

        # Parameters for lower-triangular A
        self.s1 = nn.Parameter(torch.tensor(0.0))
        self.s2 = nn.Parameter(torch.tensor(0.0))
        self.l21 = nn.Parameter(torch.tensor(0.0))

    def A(self):
        a11 = torch.exp(self.s1)
        a22 = torch.exp(self.s2)
        a21 = self.l21

        A = torch.stack([
            torch.stack([a11, torch.tensor(0.0, device=a11.device)]),
            torch.stack([a21, a22])
        ])
        return A

    def log_abs_det_A(self):
        # For triangular A, det(A) = exp(s1) * exp(s2)
        return self.s1 + self.s2

    def forward(self, z):
        """
        z -> x
        z: (N, 2)
        """
        A = self.A()
        return z @ A.T + self.mu

    def inverse(self, x):
        """
        x -> z
        x: (N, 2)
        """
        A = self.A()
        A_inv = torch.inverse(A)
        return (x - self.mu) @ A_inv.T

    def log_prob(self, x):
        """
        log p(x) = log p(z) - log|det A|
        where z = A^{-1}(x - mu)
        """
        z = self.inverse(x)

        # Standard 2D Gaussian log density
        log_pz = -0.5 * torch.sum(z**2, dim=1) - math.log(2 * math.pi)

        return log_pz - self.log_abs_det_A()

    def sample(self, n):
        z = torch.randn(n, 2, device=self.mu.device)
        return self.forward(z)


# ------------------------------------------------------------
# 3. Training loop: maximize log-likelihood with SGD
# ------------------------------------------------------------


## Exercise 2: Calculate The Jacobian

For the map $T(z) = Az + \mu$, compute the Jacobian

$$J_T(z) = \frac{\partial x}{\partial z}. $$

Questions:
1. Why is the Jacobian constant and independent of $z$?
2. Show that $J_T(z) = A$.
3. Compute $\det J_T(z)$.
4. Use the triangular structure of $A$ to simplify $\log |\det J_T(z)|$.

You can then compare your derivation with the `A()` and `log_abs_det_A()` methods in the class.


## Exercise 3: Write Down The Loss Function

The latent variable is standard Gaussian: $z \sim \mathcal{N}(0, I)$.

Using change of variables, the model density is

$$\log p_X(x) = \log p_Z\left(A^{-1}(x - \mu)\right) - \log |\det A|.$$

For a minibatch $\{x^{(i)}\}_{i=1}^B$, the training objective in this notebook is the negative log-likelihood

$$\mathcal{L}(\theta) = -\frac{1}{B} \sum_{i=1}^B \log p_X\left(x^{(i)}\right).$$

Since $p_Z$ is standard normal in 2D, each term can be written as

$$\log p_Z(z) = -\frac{1}{2}\|z\|^2 - \log(2\pi).$$

Plug this into the expression for $\log p_X(x)$ and identify exactly which part comes from the base Gaussian and which part comes from the Jacobian determinant.


In [ ]:
def train_flow(model, data, n_steps=3000, lr=1e-2, batch_size=256, device="cpu"):
    model.to(device)
    data = data.to(device)

    optimizer = torch.optim.SGD(model.parameters(), lr=lr)

    losses = []
    n = data.shape[0]

    for step in range(n_steps):
        idx = torch.randint(0, n, (batch_size,), device=device)
        batch = data[idx]

        log_px = model.log_prob(batch)
        loss = -log_px.mean()   # negative log-likelihood

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        losses.append(loss.item())

        if step % 300 == 0:
            print(f"step {step:4d} | NLL = {loss.item():.4f}")

    return losses


def gaussian_sigma_ellipse_points(mean, cov, nsig=1.0, n_points=240):
    mean = np.asarray(mean, dtype=float).reshape(2)
    cov = np.asarray(cov, dtype=float).reshape(2, 2)

    evals, evecs = np.linalg.eigh(cov)
    evals = np.maximum(evals, 1e-12)

    theta = np.linspace(0.0, 2.0 * np.pi, n_points)
    circle = np.stack([np.cos(theta), np.sin(theta)], axis=0)
    ellipse = (evecs @ (np.diag(np.sqrt(evals)) * nsig) @ circle).T + mean[None, :]
    return ellipse


@torch.no_grad()
def animate_affine_flow(model, target_mu, target_cov, n_samples=2000, n_frames=40,
                        out_gif="affine_flow_single_layer.gif", fps=12):
    device = next(model.parameters()).device
    model.eval()

    z0 = torch.randn(n_samples, 2, device=device)
    x1 = model.forward(z0)

    z_np = z0.cpu().numpy()
    x_np = x1.cpu().numpy()

    states = []
    frame_labels = []
    for t in np.linspace(0.0, 1.0, n_frames, endpoint=False):
        states.append((1.0 - t) * z_np + t * x_np)
        frame_labels.append(f"interpolation t={t:.2f}")
    states.append(x_np)
    frame_labels.append("target distribution")

    base_contours = {
        1: gaussian_sigma_ellipse_points(np.zeros(2), np.eye(2), nsig=1.0),
        2: gaussian_sigma_ellipse_points(np.zeros(2), np.eye(2), nsig=2.0),
    }
    target_contours = {
        1: gaussian_sigma_ellipse_points(target_mu, target_cov, nsig=1.0),
        2: gaussian_sigma_ellipse_points(target_mu, target_cov, nsig=2.0),
    }

    all_arrays = list(states) + list(base_contours.values()) + list(target_contours.values())
    all_pts = np.concatenate(all_arrays, axis=0)
    xmin, ymin = all_pts.min(axis=0)
    xmax, ymax = all_pts.max(axis=0)

    dx = xmax - xmin
    dy = ymax - ymin
    pad_x = 0.1 * dx if dx > 0 else 1.0
    pad_y = 0.1 * dy if dy > 0 else 1.0

    fig, ax = plt.subplots(figsize=(6, 6))
    scat = ax.scatter([], [], s=8, alpha=0.4, color="black", label="samples", zorder=3)
    title = ax.set_title("Base to target")

    for nsig, linestyle in ((1, "-"), (2, "--")):
        pts = base_contours[nsig]
        ax.plot(pts[:, 0], pts[:, 1], color="royalblue", linestyle=linestyle, linewidth=1.5,
                label=f"base {nsig}-sigma", zorder=1)
        pts = target_contours[nsig]
        ax.plot(pts[:, 0], pts[:, 1], color="darkorange", linestyle=linestyle, linewidth=1.5,
                label=f"target {nsig}-sigma", zorder=2)

    ax.set_xlim(xmin - pad_x, xmax + pad_x)
    ax.set_ylim(ymin - pad_y, ymax + pad_y)
    ax.set_xlabel("x1")
    ax.set_ylabel("x2")
    ax.set_aspect("equal")
    ax.legend(frameon=False, fontsize=9, loc="upper right")

    def init():
        scat.set_offsets(np.empty((0, 2)))
        title.set_text("Base to target")
        return scat, title

    def update(frame_idx):
        scat.set_offsets(states[frame_idx])
        title.set_text(f"Base to target | {frame_labels[frame_idx]}")
        return scat, title

    anim = FuncAnimation(
        fig,
        update,
        frames=len(states),
        init_func=init,
        blit=True,
        interval=1000 / fps,
    )
    anim.save(out_gif, writer=PillowWriter(fps=fps))
    plt.close(fig)
    print(f"Saved animation to {out_gif}")

In [ ]:
torch.manual_seed(0)
device = "cpu" 

# True data distribution
x_data, true_mu, true_cov = make_gaussian_data(n=4000, device=device)

# Model
flow = AffineFlow2D()

# Train
losses = train_flow(flow, x_data, n_steps=3000, lr=5e-2, batch_size=512, device=device)

# Learned parameters
learned_A = flow.A().detach()
learned_mu = flow.mu.detach()
learned_cov = learned_A @ learned_A.T

print("\nTrue mean:")
print(true_mu.cpu())

print("\nLearned mean:")
print(learned_mu.cpu())

print("\nTrue covariance:")
print(true_cov.cpu())

print("\nLearned covariance = A A^T:")
print(learned_cov.cpu())

# Transform data to latent space
z_data = flow.inverse(x_data).detach()

# Draw samples from learned model
x_samples = flow.sample(1000).detach()

# --------------------------------------------------------
# Plot results
# --------------------------------------------------------
fig, axes = plt.subplots(1, 3, figsize=(15, 4.5))

# Original data
axes[0].scatter(x_data[:, 0].cpu(), x_data[:, 1].cpu(), s=8, alpha=0.35)
axes[0].set_title("Training data in x-space")
axes[0].set_xlabel("x1")
axes[0].set_ylabel("x2")
axes[0].axis("equal")

# Latent space
axes[1].scatter(z_data[:, 0].cpu(), z_data[:, 1].cpu(), s=8, alpha=0.35)
axes[1].set_title("Mapped latent variables z")
axes[1].set_xlabel("z1")
axes[1].set_ylabel("z2")
axes[1].axis("equal")

# Samples from learned model
axes[2].scatter(x_data[:, 0].cpu(), x_data[:, 1].cpu(), s=8, alpha=0.2, label="data")
axes[2].scatter(x_samples[:, 0].cpu(), x_samples[:, 1].cpu(), s=8, alpha=0.35, label="flow samples")
axes[2].set_title("Data vs learned samples")
axes[2].set_xlabel("x1")
axes[2].set_ylabel("x2")
axes[2].legend()
axes[2].axis("equal")

plt.tight_layout()
plt.show()

# Loss curve
plt.figure(figsize=(6, 4))
plt.plot(losses)
plt.xlabel("Training step")
plt.ylabel("Negative log-likelihood")
plt.title("Training curve")
plt.tight_layout()
plt.show()
